In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
pwd

'c:\\Users\\User\\Documents\\brain-tumor-classification\\research'

In [3]:
os.chdir("../")

In [4]:
pwd

'c:\\Users\\User\\Documents\\brain-tumor-classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model : Path
    testing_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int


In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [8]:
class ConfigurationManager:
    def __init__(self, config_path = CONFIG_FILE_PATH, params_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self):
        return EvaluationConfig(
            path_of_model = self.config.training.trained_model_path,
            testing_data = self.config.testing.testing_data,
            all_params = self.params,
            mlflow_uri = os.getenv("MLFLOW_TRACKING_URI"),
            params_image_size = self.params.IMAGE_SIZE,
            params_batch_size = self.params.BATCH_SIZE
        )

In [9]:
import tensorflow as tf
import mlflow
import mlflow.keras
import mlflow.tensorflow
from urllib.parse import urlparse

[2026-06-13 16:30:20,048: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\brain-tumor-classification\.venv\lib\site-packages\keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.
]


In [10]:
class Evaluation:
    def __init__(self, config : EvaluationConfig):
        self.config = config 
    
    @staticmethod
    def load_model(path: Path):
        return tf.keras.models.load_model(path)
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)
    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.testing_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )
    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )

            # Workaround for MLflow 2.2.2 and TF 2.15
            import keras
            import tensorflow as tf
            if not hasattr(tf.keras, '__version__'):
                tf.keras.__version__ = keras.__version__
            mlflow.tensorflow.keras_module = keras
            
            # Model registry does not work with file store
            if tracking_url_type_store != "file":
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")

    
        

In [11]:
try:
    config = ConfigurationManager()
    eval_config = config.get_model_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.save_score()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-06-13 16:31:40,214: INFO : common : line 35 : yaml file: config\config.yaml loaded successfully]
[2026-06-13 16:31:40,245: INFO : common : line 35 : yaml file: params.yaml loaded successfully]
[2026-06-13 16:31:40,258: INFO : common : line 56 : created directory: artifacts]
[2026-06-13 16:31:47,424: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\brain-tumor-classification\.venv\lib\site-packages\keras\src\backend.py:1398: The name tf.executing_eagerly_outside_functions is deprecated. Please use tf.compat.v1.executing_eagerly_outside_functions instead.
]
[2026-06-13 16:31:48,568: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\brain-tumor-classification\.venv\lib\site-packages\keras\src\layers\pooling\max_pooling2d.py:161: The name tf.nn.max_pool is deprecated. Please use tf.nn.max_pool2d instead.
]
Found 480 images belonging to 4 classes.
[2026-06-13 16:32:00,631: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\brain-tumor

2026/06/13 16:35:59 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-06-13 16:36:03,274: INFO : builder_impl : line 801 : Assets written to: C:\Users\User\AppData\Local\Temp\tmp1ypnr3c8\model\data\model\assets]


2026/06/13 16:36:31 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\User\AppData\Local\Temp\tmp1ypnr3c8\model, flavor: tensorflow), fall back to return ['tensorflow==2.15.0']. Set logging level to DEBUG to see the full traceback.
c:\Users\User\Documents\brain-tumor-classification\.venv\lib\site-packages\_distutils_hack\__init__.py:26: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
2026/06/13 16:36:31 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/06/13 16:40:10 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 2
Created version '2' of model 'VGG16Model'.
